# Question Data Class Example

This notebook demonstrates how to instantiate a `Question` object for the L2L benchmark.

**Question**: What determines if a cell line will show MAPK6/MAPK4 pathway inhibition by Dabrafenib at 5.0uM?

The `Question.from_drug_response()` factory method automatically:
1. Finds all cell lines treated with the specified drug at the specified concentration
2. Filters to those with pathway activity data for the target pathway
3. Computes ground truth based on NES direction and FDR threshold

In [1]:
from l2l_bench import L2LData, Question
import pandas as pd

## 1. Initialize L2LData

In [2]:
data = L2LData()

HuggingFace authentication configured
Loading metadata from tahoebio/Tahoe-100M...


Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

  Gene metadata: 62710 genes


Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

  Drug metadata: 379 drugs


Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

  Cell line metadata: 102 unique cell lines


Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

  Sample metadata: 1344 samples
Metadata loaded successfully.


## 2. Create the Question

Using `Question.from_drug_response()` which auto-discovers all eligible treatments.

In [3]:
question = Question.from_drug_response(
    data=data,
    question="What determines if a cell line will show MAPK6/MAPK4 pathway inhibition by Dabrafenib at 5.0uM?",
    drug="Dabrafenib",
    concentration=5.0,
    target_pathway="MAPK6/MAPK4 Signaling R-HSA-5687128",
    positive_class_direction="repressed",  # we expect inhibition (negative NES)
    fdr_threshold=0.05
)

print("Question created successfully!")
print(f"Question: {question.question}")
print(f"Drug: {question.drug} at {question.concentration}uM")
print(f"Target pathway: {question.target_pathway}")
print(f"Positive class direction: {question.positive_class_direction}")
print(f"FDR threshold: {question.fdr_threshold}")
print(f"Test set size: {len(question.test_set)}")

Question created successfully!
Question: What determines if a cell line will show MAPK6/MAPK4 pathway inhibition by Dabrafenib at 5.0uM?
Drug: Dabrafenib at 5.0uM
Target pathway: MAPK6/MAPK4 Signaling R-HSA-5687128
Positive class direction: repressed
FDR threshold: 0.05
Test set size: 4


## 3. Examine the test set

In [5]:
# display the test set with ground truth
print("Test Set Items:\n")
print(f"{'Cell Line':<15} {'NES':>8} {'FDR':>10} {'Ground Truth':>14}")
print("-" * 50)

for item in question.test_set:
    cell_name = item.treatment.cell_line.name
    gt = "INHIBITED" if item.ground_truth else "NOT INHIBITED"
    print(f"{cell_name:<15} {item.pathway_nes:>8.3f} {item.pathway_fdr:>10.4f} {gt:>14}")

Test Set Items:

Cell Line            NES        FDR   Ground Truth
--------------------------------------------------
SW480             -1.590     0.0504  NOT INHIBITED
SHP-77            -1.320     0.3162  NOT INHIBITED
LoVo              -1.667     0.0044      INHIBITED
A549              -1.679     0.0187      INHIBITED


In [6]:
# create a summary dataframe
summary_data = []
for item in question.test_set:
    cell_line = item.treatment.cell_line
    
    # get driver mutations
    braf_mut = cell_line.has_mutation("BRAF")
    kras_mut = cell_line.has_mutation("KRAS")
    
    summary_data.append({
        "cell_line": cell_line.name,
        "organ": cell_line.organ,
        "BRAF_mutant": braf_mut,
        "KRAS_mutant": kras_mut,
        "pathway_nes": item.pathway_nes,
        "pathway_fdr": item.pathway_fdr,
        "ground_truth": item.ground_truth
    })

summary_df = pd.DataFrame(summary_data)
summary_df

,cell_line,organ,BRAF_mutant,KRAS_mutant,pathway_nes,pathway_fdr,ground_truth
0,SW480,Bowel,False,True,-1.590181,0.050406,False
1,SHP-77,Lung,False,True,-1.320339,0.316230,False
2,LoVo,Bowel,False,True,-1.667026,0.004411,True
3,A549,Lung,False,True,-1.679254,0.018675,True


## 4. Analysis: What patterns emerge?

Dabrafenib is a BRAF inhibitor. We might expect:
- BRAF-mutant cell lines to show stronger MAPK pathway inhibition
- KRAS-mutant cell lines may show resistance (KRAS is upstream of BRAF)

In [7]:
# analyze relationship between mutations and pathway inhibition
print("MAPK6/MAPK4 pathway inhibition by mutation status:\n")

print("BRAF mutants:")
braf_df = summary_df[summary_df["BRAF_mutant"]]
if len(braf_df) > 0:
    for _, row in braf_df.iterrows():
        print(f"  {row['cell_line']}: ground_truth={row['ground_truth']}, NES={row['pathway_nes']:.3f}")
else:
    print("  (none in dataset)")

print("\nKRAS mutants:")
kras_df = summary_df[summary_df["KRAS_mutant"]]
if len(kras_df) > 0:
    for _, row in kras_df.iterrows():
        print(f"  {row['cell_line']}: ground_truth={row['ground_truth']}, NES={row['pathway_nes']:.3f}")
else:
    print("  (none in dataset)")

print("\nDouble wild-type (no BRAF or KRAS mutation):")
wt_df = summary_df[~summary_df["BRAF_mutant"] & ~summary_df["KRAS_mutant"]]
if len(wt_df) > 0:
    for _, row in wt_df.iterrows():
        print(f"  {row['cell_line']}: ground_truth={row['ground_truth']}, NES={row['pathway_nes']:.3f}")
else:
    print("  (none in dataset)")

MAPK6/MAPK4 pathway inhibition by mutation status:

BRAF mutants:
  (none in dataset)

KRAS mutants:
  SW480: ground_truth=False, NES=-1.590
  SHP-77: ground_truth=False, NES=-1.320
  LoVo: ground_truth=True, NES=-1.667
  A549: ground_truth=True, NES=-1.679

Double wild-type (no BRAF or KRAS mutation):
  (none in dataset)


In [8]:
# summary statistics
n_inhibited = sum(item.ground_truth for item in question.test_set)
n_total = len(question.test_set)

print(f"\nSummary:")
print(f"  Total cell lines: {n_total}")
print(f"  Showing significant pathway inhibition: {n_inhibited} ({100*n_inhibited/n_total:.1f}%)")
print(f"  Not showing significant inhibition: {n_total - n_inhibited} ({100*(n_total-n_inhibited)/n_total:.1f}%)")


Summary:
  Total cell lines: 4
  Showing significant pathway inhibition: 2 (50.0%)
  Not showing significant inhibition: 2 (50.0%)
